# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
CONTENT = f"{BASE}/dim_content.parquet"
CLIENTS = f"{BASE}/dim_clients.parquet"

print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df())

print("\ndim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{CONTENT}')").df())

print("\ndim_clients columns:")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{CLIENTS}')").df())


fact_content_daily_performance columns:
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BI

## 1. Unit of analysis + time window

**One row means:** one content item, for one client, on one calendar day — the grain of `fact_content_daily_performance` as documented (day x client x content). This matches my lane (Refresh / Content Opportunity Scoring): I need day-level performance to detect a real trend, not a single pre-computed label like the starter CSV gave me.

**Table(s) I'll use:** `fact_content_daily_performance` (daily metrics) joined to `dim_content` (content-level attributes like word count / publish date, if present) on the content key.

**Time window:** a mid-panel month, `month=2026-03`, per the assignment's guidance — the final month (June 2026, the `_sample` file) is reserved as a sealed test month and must not be used to develop label logic, since it would double as both training data and the natural "future" outcome window.

**What I'd predict (label/proxy):** whether a content item's performance is declining within the window — computed directly from daily data this time (e.g. comparing impressions in the first half vs. second half of the month), rather than reusing a pre-baked `trend_direction` column like the starter CSV had. This is a stronger, more honest proxy because I build it myself from raw daily numbers instead of trusting an upstream label I can't fully audit.

**One thing I deliberately exclude (for now):** `fact_content_query_90d` (the query-level table). It's a different grain (content x query, not content x day) and mixing grains this early risks silently changing what a "row" means. I'll bring it in later only if I need query-level detail, and I'll re-write this contract when I do.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `impressions` | feature | direct GSC signal, knowable at report date |
| `clicks` | feature | direct GSC signal, knowable at report date |
| `avg_position` | feature | direct GSC signal, knowable at report date |
| `report_date` | context | needed to build the window/trend, not a predictive feature itself |
| `client_id` | context | needed for the client-holdout split later (per notebook 02's leakage lesson) — never a feature, since the model shouldn't key off *which* client a page belongs to |
| `content_id` | context | join key to `dim_content`, not itself predictive |
| computed trend (first-half vs second-half impressions) | **label/proxy** | this is what I'm trying to predict/rank by |
| `fact_content_query_90d` (whole table) | excluded | different grain (content x query, not content x day); mixing it in changes the unit of analysis defined in Section 1 |
| any GA4 conversion/revenue field, if present in `dim_content` or a future joined table | excluded | out of scope for a search-visibility lane; also a likely proxy for business outcomes I'm not trying to predict here |

*(Exact column names will be confirmed against the Section 0 schema printout before finalizing this table — the roles above are correct regardless of the literal column spelling.)*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries, one per claim in Section 1.

In [9]:
MONTH = "2026-03"
FACT_MONTH = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"

# Query 1 - GRAIN CHECK: one row really is one (client, content, day)?
# If the grain claim is true, grouping by these three keys should never produce a count > 1.
grain_check = con.sql(f"""
    SELECT COUNT(*) AS groups_with_duplicates
    FROM (
        SELECT client_id, content_id, report_date, COUNT(*) AS n
        FROM read_parquet('{FACT_MONTH}')
        GROUP BY client_id, content_id, report_date
        HAVING COUNT(*) > 1
    )
""").df()
print("Query 1 - grain check (should be 0 duplicate groups):")
print(grain_check)

In [ ]:
# Query 2 - ROW COUNT + DATE SPAN for month=2026-03
count_span = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_id) AS n_clients,
           COUNT(DISTINCT content_id) AS n_content_items
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("Query 2 - row count and date span for month=2026-03:")
print(count_span)


In [ ]:
# Query 3 - AVAILABILITY CHECK using IS TRUE
# Filters to rows that are actually usable (non-null, valid) - shows how many survive.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE impressions IS NOT NULL AND impressions >= 0) AS usable_impressions,
        COUNT(*) FILTER (WHERE (impressions > 0) IS TRUE) AS rows_with_any_impressions
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("Query 3 - availability check (IS TRUE filter):")
print(availability)
print(f"\nShare of rows with usable, positive impressions: "
      f"{availability['rows_with_any_impressions'][0] / availability['total_rows'][0]:.1%}")


## 4. Five features + the leakage trap

Five features built from the March 2026 slice, each with an "available when?" line, followed by the deliberate leak experiment.

In [10]:
# Build a content-level feature frame for month=2026-03:
# aggregate daily rows up to one row per (client_id, content_id) for the month.
features = con.sql(f"""
    SELECT
        client_id,
        content_id,
        SUM(impressions) AS total_impressions,                -- 1. knowable at decision moment: pure GSC history, no future data
        SUM(clicks) AS total_clicks,                           -- 2. knowable at decision moment: pure GSC history, no future data
        AVG(avg_position) AS mean_position,                    -- 3. knowable at decision moment: daily rank, already observed
        SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS ctr,-- 4. knowable at decision moment: derived only from past clicks/impressions
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN impressions ELSE 0 END)
          - SUM(CASE WHEN report_date < DATE '2026-03-16' THEN impressions ELSE 0 END)
          AS impressions_trend_delta                          -- 5. knowable at decision moment: second-half minus first-half, both already observed by month end
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY client_id, content_id
""").df()

print(f"Feature frame shape: {features.shape}")
features.head()


In [ ]:
# Define the proxy label from the same window: declining if second-half impressions < first-half
features["is_declining"] = (features["impressions_trend_delta"] < 0).astype(int)
print("Declining rate (honest proxy, built from features above):")
print(features["is_declining"].mean().round(3))


In [ ]:
# THE TRAP: add one label-derived column on purpose and watch the score jump toward perfect.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import numpy as np

base_feats = ["total_impressions", "total_clicks", "mean_position", "ctr"]
X_honest = features[base_feats].fillna(0)
y = features["is_declining"]

Xh_train, Xh_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xh_train, y_train)
honest_score = honest_tree.score(Xh_test, y_test)
print(f"HONEST accuracy (no leak): {honest_score:.3f}")

# Now sneak in impressions_trend_delta - the exact quantity the label was derived from
leaky_feats = base_feats + ["impressions_trend_delta"]
X_leaky = features[leaky_feats].fillna(0)
Xl_train, Xl_test, _, _ = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xl_train, y_train)
leaky_score = leaky_tree.score(Xl_test, y_test)
print(f"LEAKY accuracy (with impressions_trend_delta included): {leaky_score:.3f}  <- jumps toward 1.0, because")
print("this column IS the thing the label was computed from - the model isn't predicting, it's reading the answer key.")

# Delete the leaked column and keep the honest number
del X_leaky
print(f"\nKeeping the honest number going forward: {honest_score:.3f}")


## 5. Data limits

- **Unbalanced panel:** per FlyRank's own documentation, client history depth differs (`dim_clients.gsc_data_start` / `ga4_data_start`) — some clients simply have more history than others, so this slice cannot claim to represent all clients equally.
- **March 2026 is one month, one point in the yearly cycle:** any seasonal SEO effects (e.g. holiday-driven traffic patterns) specific to March won't generalize automatically to other months without re-checking.
- **The final month (June 2026 / `_sample`) is sealed:** I deliberately did not look at it while building label logic, since it's the natural "future" window for any past-to-future label — using it now would let information leak backward into my feature design.
- **This data cannot tell me *why* a page's trend changed** — only that it changed. Any causal story (e.g. "a refresh caused recovery") would require an actual experiment, not this observational daily-performance data.
- **Query-level detail is excluded from this slice:** because `fact_content_query_90d` was deliberately left out (Section 1), this contract cannot yet say anything about *which* search queries drove the impressions/clicks I'm aggregating.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.